# 12 · Shortest-path network interdiction (a bilevel game made single-level)

**Goal:** model an attacker–defender (interdictor–evader) game, turn the max–min into a single MILP **using LP duality**, and verify it by brute force.

The interdictor picks arcs to delay ($x_{ij}\in\{0,1\}$, budget $\sum b_{ij}x_{ij}\le B$). The evader then takes the shortest path under lengths $c_{ij}+d_{ij}x_{ij}$:

$$\max_{x\in X}\ \min_{y\ \text{an } s\text{-}t\text{ path}}\ \sum_{(i,j)}(c_{ij}+d_{ij}x_{ij})\,y_{ij}.$$

**Step 1.** For fixed $x$ the inner problem is the shortest-path LP. **Step 2.** Replace it with its dual from notebook 08: $\max\ \pi_t-\pi_s$ s.t. $\pi_j-\pi_i\le c_{ij}+d_{ij}x_{ij}$. **Step 3.** max-of-max is one max:

$$\max_{\pi,x}\ \pi_t\quad\text{s.t.}\quad \pi_j-\pi_i-d_{ij}x_{ij}\le c_{ij},\quad b^\top x\le B,\quad \pi_s=0,\ x\in\{0,1\}^m .$$

It's linear because $d_{ij}x_{ij}$ multiplies data by a variable, never two variables (Israeli & Wood, 2002).

In [ ]:
import itertools
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
from scipy.optimize import milp, LinearConstraint, Bounds

In [ ]:
import heapq

def dijkstra(n, arcs, s):
    # arcs: list of (i, j, length) with length >= 0. Returns distances and predecessor array.
    adj = [[] for _ in range(n)]
    for i, j, c in arcs:
        adj[i].append((j, c))
    dist = np.full(n, np.inf); dist[s] = 0.0
    pred = [None] * n
    done = [False] * n
    pq = [(0.0, s)]
    while pq:
        d, u = heapq.heappop(pq)
        if done[u]:
            continue
        done[u] = True                       # u's label is now permanent
        for v, c in adj[u]:
            if d + c < dist[v]:
                dist[v] = d + c; pred[v] = u
                heapq.heappush(pq, (dist[v], v))
    return dist, pred

def path_to(pred, t):
    path = [t]
    while pred[path[-1]] is not None:
        path.append(pred[path[-1]])
    return path[::-1]

In [ ]:
# A small supply network: s = 0 (depot), t = 7 (destination).
# Each arc: (i, j, length c, extra delay d if interdicted, interdiction cost b)
ARCS = [
    (0, 1, 3, 8, 2), (0, 2, 4, 8, 2), (0, 3, 6, 8, 2),
    (1, 4, 4, 8, 1), (1, 2, 2, 8, 1), (2, 4, 3, 8, 1), (2, 5, 5, 8, 1),
    (3, 5, 3, 8, 1), (3, 6, 7, 8, 1), (4, 7, 6, 8, 1), (4, 5, 2, 8, 1),
    (5, 7, 4, 8, 1), (5, 6, 2, 8, 1), (6, 7, 3, 8, 2),
]
n, s, t = 8, 0, 7
m = len(ARCS)
c = np.array([a[2] for a in ARCS], float)
d = np.array([a[3] for a in ARCS], float)
b = np.array([a[4] for a in ARCS], float)
POS = {0: (0, 1), 1: (1, 2), 2: (1, 1), 3: (1, 0), 4: (2, 2), 5: (2, 1), 6: (2, 0), 7: (3, 1)}

In [ ]:
base = [(i, j, ci) for (i, j, ci, _, _) in ARCS]
dist0, pred0 = dijkstra(n, base, s)
print("no interdiction: shortest length", dist0[t], " path", path_to(pred0, t))

## The single-level MILP

In [ ]:
def interdict(B):
    # variables z = [pi (n), x (m)]
    obj = np.zeros(n + m); obj[t] = -1.0                       # maximize pi_t
    A = np.zeros((m, n + m))
    for k, (i, j, *_) in enumerate(ARCS):
        A[k, j] = 1.0; A[k, i] = -1.0; A[k, n + k] = -d[k]     # pi_j - pi_i - d x <= c
    cons = [LinearConstraint(A, -np.inf, c),
            LinearConstraint(np.r_[np.zeros(n), b][None, :], -np.inf, B)]
    lb = np.zeros(n + m)
    ub = np.r_[np.full(n, np.inf), np.ones(m)]
    ub[s] = 0.0                                                # pi_s = 0
    integrality = np.r_[np.zeros(n), np.ones(m)]
    res = milp(obj, constraints=cons, integrality=integrality, bounds=Bounds(lb, ub))
    return -res.fun, np.round(res.x[n:]).astype(int)

B = 3
val, x = interdict(B)
hit = [ARCS[k][:2] for k in range(m) if x[k]]
print(f"budget {B}: interdict {hit}  -> evader's shortest length {val}")

## Verify by brute force

Enumerate every interdiction plan within budget, run Dijkstra on the modified lengths, and keep the worst case for the evader.

In [ ]:
def brute_force(B):
    best_val, best_set = -np.inf, None
    for r in range(m + 1):
        for S_ in itertools.combinations(range(m), r):
            if b[list(S_)].sum() > B:
                continue
            lengths = c.copy(); lengths[list(S_)] += d[list(S_)]
            dd, _ = dijkstra(n, [(ARCS[k][0], ARCS[k][1], lengths[k]) for k in range(m)], s)
            if dd[t] > best_val:
                best_val, best_set = dd[t], S_
    return best_val, best_set

bf_val, bf_set = brute_force(B)
print("brute force:", bf_val, [ARCS[k][:2] for k in bf_set])
assert np.isclose(bf_val, val)

## The evader's response and the value of budget

In [ ]:
lengths = c + d * x
dist1, pred1 = dijkstra(n, [(ARCS[k][0], ARCS[k][1], lengths[k]) for k in range(m)], s)
p1 = path_to(pred1, t)
print("evader's new path:", p1, " length", dist1[t])

budgets = range(0, 7)
vals = [interdict(Bk)[0] for Bk in budgets]
for Bk in (1, 2):
    assert np.isclose(interdict(Bk)[0], brute_force(Bk)[0])

G = nx.DiGraph([(i, j) for i, j, *_ in ARCS])
fig, axs = plt.subplots(1, 2, figsize=(11, 3.8))
ecol = ["C3" if x[k] else "0.7" for k in range(m)]
on_p = set(zip(p1[:-1], p1[1:]))
wid = [3.5 if (i, j) in on_p else 1.2 for i, j, *_ in ARCS]
nx.draw_networkx(G, POS, ax=axs[0], edgelist=[a[:2] for a in ARCS], edge_color=ecol,
                 width=wid, node_color="w", edgecolors="k")
nx.draw_networkx_edge_labels(G, POS, ax=axs[0], edge_labels={(a[0], a[1]): a[2] for a in ARCS}, font_size=8)
axs[0].set_title(f"B={B}: red = interdicted, thick = evader's path"); axs[0].axis("off")
axs[1].step(list(budgets), vals, where="post", marker="o")
axs[1].set_xlabel("interdiction budget B"); axs[1].set_ylabel("evader's shortest length")
axs[1].set_title("value of interdiction budget")
plt.tight_layout(); plt.show()

## What to take away

* **Dualize the inner problem** to turn a bilevel max–min into one MILP. This works whenever the inner problem is an LP (shortest path, max flow, min-cost flow).
* The curve of value against budget has diminishing returns. Once the cheap bottlenecks are hit, the evader has good alternatives.
* For large networks the MILP gets hard. **Benders decomposition** and cut-generation methods are the standard next step.

## Exercises

1. Make interdiction *destroy* arcs: set $d_{ij}$ to a big-$M$ larger than any simple path length. How large must $M$ be, and what does a large $M$ do to the LP relaxation?
2. **Max-flow interdiction:** dualize the max-flow LP from notebook 09 into a min-cut LP and write the interdictor's MILP. Compare with `max_flow_interdiction/` in this repo.
3. Implement a simple **Benders** loop: the master picks $x$, the subproblem (Dijkstra) returns the evader's path $P$, add the cut $\eta\le\sum_{(i,j)\in P}(c_{ij}+d_{ij}x_{ij})$, and repeat.